In [1]:
import os
import fitz
import requests
import hashlib
import pickle
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction


/Users/ashrita/Desktop/rag_app/rag-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L12-v2"
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)


In [3]:
def extract_text_from_pdf(path):
    doc = fitz.open(path)
    return "\n".join([page.get_text() for page in doc])

def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]

def get_pdf_hash(filepaths):
    m = hashlib.md5()
    for path in sorted(filepaths):
        with open(path, 'rb') as f:
            m.update(f.read())
    return m.hexdigest()

def generate_answer_with_ollama(context, query):
    prompt = f"""You are an AI assistant. Use only the context provided below to answer the user's question.

If the answer is not present in the context, say "I don't know".

Context:
{context}

Question: {query}

Answer:"""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "mistral", "prompt": prompt, "stream": False}
    )
    return response.json()["response"].strip()


In [4]:
file_paths = [os.path.join("data", f) for f in os.listdir("data") if f.endswith(".pdf")]
all_chunks = []
for pdf_path in file_paths:
    text = extract_text_from_pdf(pdf_path)
    all_chunks.extend(chunk_text(text))


In [5]:
pdf_hash = get_pdf_hash(file_paths)
cache_path = os.path.join(".cache", f"{pdf_hash}.pkl")
os.makedirs(".cache", exist_ok=True)

if os.path.exists(cache_path):
    with open(cache_path, "rb") as f:
        ids, cached_chunks = pickle.load(f)
else:
    ids = [f"chunk_{i}" for i in range(len(all_chunks))]
    cached_chunks = all_chunks
    with open(cache_path, "wb") as f:
        pickle.dump((ids, cached_chunks), f)


In [6]:
chroma_client = PersistentClient(path=".chroma")
embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)
collection = chroma_client.get_or_create_collection(name="pdf_chunks_mistral_local", embedding_function=embed_fn)

if len(collection.get()["ids"]) == 0:
    collection.add(documents=cached_chunks, ids=ids)
    print(" Stored chunks in Chroma.")
else:
    print("Chroma collection already populated.")


✅ Chroma collection already populated.


In [10]:
query = input("Enter your question: ")

results = collection.query(query_texts=[query], n_results=3)
top_chunks = results['documents'][0]
context = "\n".join(top_chunks)

answer = generate_answer_with_ollama(context, query)

print("\n Retrieved Context:")
print(context)

print("\n Answer:")
print(answer if answer.strip() != "." else "Sorry, I couldn't find a relevant answer in the documents.")



📄 Retrieved Context:
fact you have features like Data lineage and auditing that are important, to track the modifications made in your data and get the details of who, how and when it was changed. Security tools like Apache Ranger, Apache Atlas, or AWS Lake Formation assist organizations in implementing security policies and preserving data integrity. The increased focus on data governance has made it a key responsibility of data engineers. g. The Cloud Data Engineering Takeover If you are not aware, cloud computing has transformed data engineering with immediate scalability, convenient pricing and integrated data. Data is stored, processed, and analyzed in any cloud-based data platform such as AWS, Google Cloud, and Microsoft Azure, which organizations are increasingly implementing. Cross-Cloud Data ManagerCloud data lakes and warehouses such as Amazon S3, Google BigQuery, and Azure Synapse Analytics makes it easier for companies to use on-demand databases to administer huge datasets